# Checkpoint II · Examen blanc — notebook de l'examen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_2/02_examen_notebook.ipynb)

Ce notebook accompagne le sujet `01_examen_sujet.md`.

- **Partie A, pendant l'examen** : les deux questions de code, CP2.10 (🐛, 1,5 point) et CP2.11 (🔨, 1 point). Écris ton code dans les cellules à compléter, sans `mylearn`, sans scikit-learn, sans les corrigés et sans assistant IA ; `help()` et la documentation officielle de NumPy sont permis. Rien n'est vérifié pendant l'examen : les cellules « essai » montrent seulement ce que fait ton code. Tes réponses chiffrées et tes explications vont sur ta feuille (`04_mes_reponses.md`).
- **Partie B, après l'examen** : la vérification automatique de ton code et des réponses de ta feuille. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`.

« Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳ (partie A) ou ⏸️ (partie B).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_2/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP2`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · Pendant l'examen : les deux questions de code

In [ ]:
# Data and tools of the two code questions (run this cell first)
import numpy as np

california = wb.datasets.load_california()                    # 20 640 districts (1990 census)
FEATURES = california.columns[:8].tolist()                    # MedInc, HouseAge, ..., Latitude, Longitude
rows = np.sort(np.random.default_rng(2610).choice(len(california), 2000, replace=False))
X = california[FEATURES].to_numpy(dtype=float)[rows]          # shape (2000, 8)
y = california["MedHouseVal"].to_numpy(dtype=float)[rows]     # median house value (hundreds of thousands of $)


def make_cells(size):
    """Id of the square cell (side `size`, in degrees) of the latitude-longitude grid that holds each district."""
    row = np.floor(np.round(X[:, 6] / size, 8)).astype(int)    # latitude band
    col = np.floor(np.round(X[:, 7] / size, 8)).astype(int)    # longitude band
    return row * 100_000 + col


CELLS = {size: make_cells(size) for size in (0.5, 0.1, 0.05)}  # the three grids of CP2.10 d)


def cell_means(cells, y):
    """Dict: cell id -> mean of y over the districts of that cell (only the districts passed)."""
    return {cell: float(y[cells == cell].mean()) for cell in np.unique(cells)}


def lookup(cells, means, default):
    """For each district, the mean of its cell in `means`, or `default` if its cell is not in `means`."""
    return np.array([means.get(cell, default) for cell in cells], dtype=float)


def kfold(n, k=5, seed=0):
    """List of k pairs (train_indices, val_indices): the n indices, shuffled once, cut into k folds."""
    folds = np.array_split(np.random.default_rng(seed).permutation(n), k)
    return [(np.concatenate([folds[j] for j in range(k) if j != i]), folds[i]) for i in range(k)]


def ridge_fit(F, y, alpha):
    """Ridge regression with an intercept that is not penalized: returns (w, b)."""
    F_mean, y_mean = F.mean(axis=0), y.mean()
    Fc = F - F_mean
    w = np.linalg.solve(Fc.T @ Fc + alpha * np.eye(F.shape[1]), Fc.T @ (y - y_mean))
    return w, y_mean - F_mean @ w


def ridge_predict(model, F):
    w, b = model
    return F @ w + b


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


print("X:", X.shape, "· y:", y.shape, "· cells of 0.1°:", len(np.unique(CELLS[0.1])))

### CP2.10 — La fuite cachée d'une validation croisée 🐛 ★ ⏱️ 9 min · 1,5 point
**Objectif :** trouver ce qu'une validation croisée a appris de ses folds de validation, et la corriger.

Un collègue estime la RMSE d'une régression Ridge par validation croisée à 5 folds, sur les 2 000 districts de `X` et `y`. En plus des huit features, il ajoute « le prix du quartier » : la grille latitude-longitude est découpée en cases de 0,1° de côté (`CELLS[0.1]` donne la case de chaque district), et chaque district reçoit la moyenne des prix (`y`) des districts de sa case. Il standardise ensuite les neuf colonnes, puis lance la validation croisée. Exécute son code (cellule suivante).

a) Deux étapes de son code apprennent quelque chose des données avant la boucle de validation croisée. Lesquelles ? Laquelle fausse vraiment son score, et pourquoi l'autre ne le change presque pas ? (sur ta feuille) **[0,4 pt]**
b) Écris `cv_rmse_fixed(X, y, cells, k=5, alpha=1.0)`, qui fait le même calcul **sans fuite**. Dans chaque tour (`kfold(len(y), k)`, comme chez le collègue) : le prix du quartier de chaque district, d'entraînement comme de validation, est la moyenne des prix des districts **d'entraînement du tour** qui sont dans sa case ; un district dont la case ne contient aucun district d'entraînement reçoit la moyenne des prix d'entraînement du tour ; la standardisation utilise la moyenne et l'écart-type des lignes d'entraînement du tour. La fonction renvoie la moyenne des $k$ RMSE de validation, sans modifier ses arguments. **[0,6 pt]**
c) La RMSE moyenne de ta fonction avec les cases de 0,1° (3 décimales, sur ta feuille). **[0,2 pt]**
d) Le collègue a aussi comparé des cases de 0,5°, 0,1° et 0,05° avec son code (sortie de sa cellule), et garde les plus petites. Compare avec ta fonction : quelle taille garderais-tu ? Pourquoi la fuite le poussait-elle vers les plus petites cases ? (sur ta feuille) **[0,3 pt]**

In [ ]:
# The colleague's code (do not change this cell: run it and read what it prints)
def cv_rmse_colleague(X, y, cells, k=5, alpha=1.0):
    neighbourhood = lookup(cells, cell_means(cells, y), y.mean())   # "price of the neighbourhood" of each district
    F = np.column_stack([X, neighbourhood])
    F = (F - F.mean(axis=0)) / F.std(axis=0)                         # standardize the 9 columns
    scores = []
    for train, val in kfold(len(y), k):
        model = ridge_fit(F[train], y[train], alpha)
        scores.append(rmse(y[val], ridge_predict(model, F[val])))
    return float(np.mean(scores))


for size, cells in CELLS.items():
    print(f"cells of {size}°: the colleague's cross-validated RMSE = {cv_rmse_colleague(X, y, cells):.4f}")

In [ ]:
def cv_rmse_fixed(X, y, cells, k=5, alpha=1.0):
    """Mean validation RMSE of the colleague's model over k folds, without leakage (see CP2.10 b)."""
    raise NotImplementedError("cv_rmse_fixed")  # TODO CP2.10 b)

In [ ]:
# Try your function (no check during the exam: compare with what you expect)
with wb.attempt("CP2.10"):
    for size, cells in CELLS.items():
        print(f"cells of {size}°: cross-validated RMSE without leakage = {cv_rmse_fixed(X, y, cells):.4f}")

### CP2.11 — Coder `epsilon_greedy_action` et une moyenne incrémentale 🔨 ★ ⏱️ 9 min · 1 point
**Objectif :** programmer, sans `mylearn`, les deux briques d'un agent de bandit.

a) Écris `epsilon_greedy_action(q_values, epsilon, rng)`. Avec la probabilité $\varepsilon$ (un tirage `rng.random() < epsilon`), elle explore : un bras au hasard parmi **tous** les bras, `rng.integers(K)` ($K$ = nombre de bras). Sinon, elle exploite : un bras d'estimation maximale, tiré au hasard parmi les ex aequo avec `rng.choice`. Elle renvoie un `int` et ne modifie pas `q_values`. **[0,5 pt]**
b) Écris `incremental_estimates(rewards, step=None)`, qui renvoie la liste des estimations successives d'un bras, une après chaque récompense, en partant de $Q = 0$ : à la $n$-ième récompense $R$, $Q \leftarrow Q + a\,(R - Q)$, avec $a = 1/n$ si `step` vaut `None` (la moyenne exacte), et $a$ = `step` sinon (un pas constant). La fonction ne recalcule jamais la somme de toutes les récompenses. **[0,3 pt]**
c) Avec ta fonction, la dernière estimation pour les récompenses `[1, 0, 0, 1, 1, 1, 0, 1]` et un pas constant de 0,25 (4 décimales, sur ta feuille). **[0,2 pt]**

In [ ]:
def epsilon_greedy_action(q_values, epsilon, rng):
    """The arm chosen by the epsilon-greedy rule (an int), see CP2.11 a)."""
    raise NotImplementedError("epsilon_greedy_action")  # TODO CP2.11 a)


def incremental_estimates(rewards, step=None):
    """The list of the successive estimates Q, one after each reward, starting from Q = 0 (CP2.11 b)."""
    raise NotImplementedError("incremental_estimates")  # TODO CP2.11 b)

In [ ]:
# Try your functions (no check during the exam)
with wb.attempt("CP2.11"):
    rng_try = np.random.default_rng(0)
    print("ten choices with epsilon = 0.2:", [epsilon_greedy_action([0.2, 0.9, 0.5, 0.9], 0.2, rng_try) for _ in range(10)])
with wb.attempt("CP2.11"):
    print("estimates, exact mean   :", incremental_estimates([1, 0, 0, 1, 1, 1, 0, 1]))
    print("estimates, step of 0.25 :", incremental_estimates([1, 0, 0, 1, 1, 1, 0, 1], step=0.25))

---

### 🛑 Fin de la partie A

L'examen s'arrête ici. Rends ta copie (enregistre ce notebook et ta feuille), puis, seulement après, passe à la partie B.

## Partie B · Après l'examen : vérification automatique

**Ne commence cette partie qu'une fois ta copie terminée**, examen rendu. Elle vérifie le code de la partie A, puis les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Elle ne note pas : les démarches, les justifications et les questions rédigées se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Exécute la vérification du code (CP2.10 et CP2.11).
3. Reporte ensuite chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). En Python, le séparateur décimal est un **point** (`0.125`), et les lettres s'écrivent entre guillemets (`"XYZ"`).

Si tu as fermé le notebook depuis l'examen, exécute d'abord la cellule de configuration (tout en haut) et les cellules de la partie A, sauf les essais.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells of part B

RESULTS = {}        # sub-ID -> result of wb.check
PROPERTIES = {}     # question -> [(property, passed)], reset each time its check cell runs
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans la première cellule de la partie B.")
        _REMINDED.append(True)
        return False
    if "wb" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


def ready(question, *names):
    """True when the cells of part A that define `names` have run (after a kernel restart, run them again)."""
    missing = [name for name in names if name not in globals()]
    if missing:
        verb = "n'existe" if len(missing) == 1 else "n'existent"
        print(f"⚠️ {question} : exécute d'abord les cellules de la partie A ({', '.join(missing)} {verb} pas encore).")
    return not missing


def verdict(question, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message, and remember the result."""
    PROPERTIES.setdefault(question, []).append((success or failure, bool(ok)))
    print(f"✅ {question} : {success}" if ok else f"❌ {question} : {failure}")
    return bool(ok)


_LIBRARY_ADVICE = " ; si elle fait partie de `mylearn`, ses tests (`pytest`) disent quel cas pose problème."


def code_check(ex_id, value, reminder=""):
    """wb.check of a value computed by your exam function (no advice about mylearn: it is not used here)."""
    result = wb.check(ex_id, value, computed=True, quiet=True)
    message = result.message.replace(_LIBRARY_ADVICE, ".")
    if not result and reminder and not message.startswith("Erreur classique"):
        message += f" Rappel : {reminder}"
    print(f"{'✅' if result else '❌'} Ex {ex_id} : {message}")
    RESULTS[ex_id] = result
    return result


def as_letters(value):
    """Letters typed as "ABCD", "A, B, C, D" or ["A", "B", "C", "D"] -> "ABCD" (`...` stays `...`)."""
    if value is ... or value is None:
        return value
    if isinstance(value, (list, tuple)):
        value = "".join(str(v) for v in value)
    return "".join(ch for ch in str(value) if ch.isalpha()).upper()


def as_float(value):
    """A number returned by a function, as a float (None when it is not one)."""
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    return number if np.isfinite(number) else None

### Le code de la partie A

In [ ]:
# Checks of cv_rmse_fixed (CP2.10)
if exam_over() and ready("CP2.10", "np", "X", "y", "CELLS", "cell_means", "lookup", "kfold", "ridge_fit", "ridge_predict",
                         "rmse", "cv_rmse_colleague", "cv_rmse_fixed"):
    PROPERTIES["CP2.10"] = []
    for sub in ("CP2.10c", "CP2.10d"):
        RESULTS.pop(sub, None)
    X_check, y_check = X.copy(), y.copy()
    cells_check = {size: cells.copy() for size, cells in CELLS.items()}
    fitted_on = []                             # the matrices that cv_rmse_fixed gives to ridge_fit
    given_ridge_fit = ridge_fit

    def ridge_fit(F, y, alpha):                # the same ridge_fit (same parameter names), which keeps a copy of F
        fitted_on.append(np.array(F, dtype=float, copy=True))
        return given_ridge_fit(F, y, alpha)

    values = None
    try:
        values = [cv_rmse_fixed(X_check, y_check, cells_check[size]) for size in (0.5, 0.1, 0.05)]
    except NotImplementedError:
        print("⏳ CP2.10 : cv_rmse_fixed n'est pas encore écrite.")
    except Exception as exc:                   # a bug of the exam code must not stop part B
        verdict("CP2.10", False, "", f"cv_rmse_fixed(X, y, cells) lève une erreur : {type(exc).__name__}: {exc}")
    finally:
        ridge_fit = given_ridge_fit
    if values is not None:
        unchanged = (np.array_equal(X_check, X) and np.array_equal(y_check, y)
                     and all(np.array_equal(cells_check[size], CELLS[size]) for size in CELLS))
        verdict("CP2.10", unchanged, "X, y et cells ne sont pas modifiés.",
                "ta fonction modifie X, y ou cells : calcule de nouveaux tableaux (np.column_stack, puis une "
                "standardisation qui crée un nouveau tableau) au lieu d'écrire dans ses arguments.")
        numbers = []
        for value in values:
            try:
                numbers.append(float(value))
            except (TypeError, ValueError):
                numbers.append(None)
        if any(v is None for v in values):
            verdict("CP2.10", False, "", "cv_rmse_fixed ne renvoie rien : as-tu oublié le return ?")
        elif any(n is None for n in numbers):
            verdict("CP2.10", False, "", f"cv_rmse_fixed doit renvoyer un nombre (la moyenne des k RMSE), pas "
                    f"un objet de type {type(values[0]).__name__}.")
        elif not all(np.isfinite(numbers)):
            verdict("CP2.10", False, "", "cv_rmse_fixed renvoie NaN ou l'infini : un district dont la case ne "
                    "contient aucun district d'entraînement du tour reçoit-il bien une valeur par défaut ?")
        else:
            n_train = len(kfold(len(y), 5)[0][0])
            if fitted_on:
                standardized = all(F.shape == (n_train, X.shape[1] + 1) and np.allclose(F.mean(axis=0), 0, atol=1e-6)
                                   and np.allclose(F.std(axis=0), 1, atol=1e-2) for F in fitted_on)
                verdict("CP2.10", standardized,
                        "ridge_fit reçoit les lignes d'entraînement du tour, standardisées avec leurs propres "
                        "moyennes et écarts-types.",
                        f"ridge_fit doit recevoir les {n_train} lignes d'entraînement du tour et leurs neuf colonnes, "
                        f"standardisées avec la moyenne et l'écart-type de ces lignes-là (chaque colonne de moyenne "
                        f"0 et d'écart-type 1).")
            else:
                print("ℹ️ CP2.10 : ta fonction n'appelle pas ridge_fit ; la standardisation n'est pas vérifiée.")
            leaky = cv_rmse_colleague(X, y, CELLS[0.1])
            if numbers[1] < leaky - 0.02:
                verdict("CP2.10", False, "", "ta valeur est nettement plus basse que la RMSE du collègue, alors que "
                        "supprimer sa fuite fait remonter l'erreur : une fuite plus forte reste-t-elle (le prix d'un "
                        "district de validation dans sa propre feature) ? Et renvoies-tu bien la moyenne des k RMSE "
                        "(des racines, pas des carrés) ?")
            else:
                verdict("CP2.10", numbers[1] > leaky + 0.05,
                        "sans la fuite du prix du quartier, la RMSE est nettement plus haute que celle du collègue.",
                        "ta RMSE reste trop près de celle du collègue : le prix du quartier des districts de "
                        "validation doit venir des seuls districts d'entraînement du tour.")
            reminder = ("relis l'énoncé de b) : dans chaque tour, d'où viennent les moyennes par case, la valeur par "
                        "défaut et la standardisation ?")
            code_check("CP2.10c", numbers[1], reminder)
            code_check("CP2.10d", numbers, reminder)

In [ ]:
# Checks of epsilon_greedy_action and incremental_estimates (CP2.11)
if exam_over() and ready("CP2.11", "np", "epsilon_greedy_action", "incremental_estimates"):
    PROPERTIES["CP2.11"] = []
    RESULTS.pop("CP2.11c", None)

    def choices(q, epsilon, n, seed):
        """n choices with a fresh copy of q at each call (a function that changes q must not skew the counts)."""
        rng_check = np.random.default_rng(seed)
        return [epsilon_greedy_action(list(q), epsilon, rng_check) for _ in range(n)]

    try:
        first = epsilon_greedy_action([0.2, 0.9, 0.5], 0.0, np.random.default_rng(0))
    except NotImplementedError:
        print("⏳ CP2.11 a) : epsilon_greedy_action n'est pas encore écrite.")
    except Exception as exc:                   # a bug of the exam code must not stop part B
        verdict("CP2.11", False, "", f"epsilon_greedy_action lève une erreur : {type(exc).__name__}: {exc}")
    else:
        if first is None:
            verdict("CP2.11", False, "", "epsilon_greedy_action ne renvoie rien : as-tu oublié le return ?")
        else:
            try:
                explored = epsilon_greedy_action([0.2, 0.9, 0.5], 1.0, np.random.default_rng(0))
                kinds = [type(v).__name__ for v in (first, explored)]
                verdict("CP2.11", all(isinstance(v, int) and not isinstance(v, bool) for v in (first, explored)),
                        "elle renvoie un int, en exploitant comme en explorant.",
                        f"elle doit renvoyer un int Python, en exploitant comme en explorant (ici : {kinds[0]} et "
                        f"{kinds[1]}) : return int(...) (le reste est vérifié quand même).")
                q_list, q_array = [0.2, 0.9, 0.5], np.array([0.2, 0.9, 0.5])
                for seed in range(20):
                    epsilon_greedy_action(q_list, 0.5, np.random.default_rng(seed))
                    epsilon_greedy_action(q_array, 0.5, np.random.default_rng(seed))
                verdict("CP2.11", q_list == [0.2, 0.9, 0.5] and q_array.tolist() == [0.2, 0.9, 0.5],
                        "q_values n'est pas modifié.",
                        "ta fonction modifie q_values : elle doit seulement le lire.")
                greedy = choices([0.2, 0.9, 0.5], 0.0, 300, 1)
                verdict("CP2.11", set(map(int, greedy)) == {1}, "avec ε = 0, toujours le meilleur bras.",
                        "avec ε = 0, elle doit toujours renvoyer le bras d'estimation maximale (ici 1).")
                tied = np.array(choices([0.5, 0.9, 0.9, 0.1], 0.0, 4000, 2), dtype=int)
                share = np.mean(tied == 1)
                verdict("CP2.11", set(tied.tolist()) <= {1, 2} and 0.45 < share < 0.55,
                        "les ex aequo sont tirés au hasard, à parts égales.",
                        "avec ε = 0 et deux maxima (les bras 1 et 2), chacun doit sortir environ une fois sur deux "
                        "(rng.choice parmi les bras d'estimation maximale ; np.argmax prend toujours le premier).")
                explore = np.array(choices([0.0, 1.0, 0.0, 0.0], 1.0, 8000, 3), dtype=int)
                counts = np.bincount(explore, minlength=4)
                verdict("CP2.11", len(counts) == 4 and np.all(np.abs(counts / 8000 - 0.25) < 0.02),
                        "avec ε = 1, chacun des quatre bras sort environ une fois sur quatre.",
                        f"avec ε = 1, l'exploration tire un bras parmi TOUS (rng.integers(K)) ; fréquences obtenues : "
                        f"{np.round(counts / 8000, 3).tolist()}.")
                mixed = np.array(choices([0.1, 0.3, 0.8, 0.2, 0.4], 0.2, 20000, 4), dtype=int)
                share_best = np.mean(mixed == 2)
                verdict("CP2.11", abs(share_best - 0.84) < 0.015,
                        "avec ε = 0,2 et cinq bras, le meilleur sort environ 84 % du temps (1 − ε + ε/K).",
                        f"avec ε = 0,2 et cinq bras, le meilleur devrait sortir 1 − ε + ε/K = 84 % du temps, pas "
                        f"{share_best * 100:.1f} %".replace(".", ",") + " : l'exploration se décide avec "
                        "rng.random() < epsilon, et inclut le meilleur bras.")
                verdict("CP2.11", choices([0.3, 0.3, 0.1], 0.5, 50, 5) == choices([0.3, 0.3, 0.1], 0.5, 50, 5),
                        "même générateur, mêmes choix.",
                        "deux générateurs de même graine doivent donner les mêmes choix : n'utilise que le rng reçu.")
            except Exception as exc:
                verdict("CP2.11", False, "", f"epsilon_greedy_action lève une erreur : {type(exc).__name__}: {exc}")
    rewards_check = [1, 0, 0, 1, 1, 1, 0, 1]
    try:
        means_out = incremental_estimates(list(rewards_check))
        steps_out = incremental_estimates(list(rewards_check), step=0.25)
    except NotImplementedError:
        print("⏳ CP2.11 b) : incremental_estimates n'est pas encore écrite.")
    except Exception as exc:
        verdict("CP2.11", False, "", f"incremental_estimates lève une erreur : {type(exc).__name__}: {exc}")
    else:
        means_arr = steps_arr = None
        if means_out is None or steps_out is None:
            verdict("CP2.11", False, "", "incremental_estimates ne renvoie rien : as-tu oublié le return ?")
        else:
            try:
                means_arr = np.asarray(list(means_out), dtype=float).ravel()     # list(): a generator works too
                steps_arr = np.asarray(list(steps_out), dtype=float).ravel()
            except (TypeError, ValueError):
                means_arr = steps_arr = None
                verdict("CP2.11", False, "", f"incremental_estimates doit renvoyer la liste des estimations, une par "
                        f"récompense, pas un objet de type {type(steps_out).__name__}.")
        if means_arr is not None:
            exact = np.cumsum(rewards_check) / np.arange(1, 9)
            verdict("CP2.11", means_arr.shape == (8,) and np.allclose(means_arr, exact),
                    "sans pas constant, les estimations sont les moyennes successives.",
                    "sans pas constant (step=None), la n-ième estimation doit être la moyenne des n premières "
                    "récompenses : Q ← Q + (R − Q)/n, en partant de Q = 0, une estimation par récompense.")
            if steps_arr.shape == (8,):
                code_check("CP2.11c", float(steps_arr[-1]),
                           "Q part de 0, puis Q ← Q + 0,25 (R − Q) à chaque récompense, dans l'ordre.")
            else:
                verdict("CP2.11", False, "", f"incremental_estimates doit renvoyer une estimation par récompense "
                        f"(8 ici), pas {steps_arr.size}.")

### Les réponses de ta feuille

**CP2.1 — Vrai ou faux justifiés (le verdict seulement ; les justifications se notent avec le corrigé)**

In [ ]:
# CP2.1: the values written on your answer sheet
answer_CP2_1a = ...  # a) True / False (ou "vrai" / "faux")
answer_CP2_1b = ...  # b) True / False (ou "vrai" / "faux")
answer_CP2_1c = ...  # c) True / False (ou "vrai" / "faux")
answer_CP2_1d = ...  # d) True / False (ou "vrai" / "faux")
answer_CP2_1e = ...  # e) True / False (ou "vrai" / "faux")
answer_CP2_1f = ...  # f) True / False (ou "vrai" / "faux")
answer_CP2_1g = ...  # g) True / False (ou "vrai" / "faux")
answer_CP2_1h = ...  # h) True / False (ou "vrai" / "faux")

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP2_1a, answer_CP2_1b, answer_CP2_1c, answer_CP2_1d, answer_CP2_1e, answer_CP2_1f, answer_CP2_1g, answer_CP2_1h]):
        RESULTS[f"CP2.1{letter}"] = wb.check(f"CP2.1{letter}", answer)

**CP2.2 — Un-contre-tous, un-contre-un**

In [ ]:
# CP2.2: the values written on your answer sheet
answer_CP2_2a = ...  # a) [N_OvR, N_OvO] for K = 12
answer_CP2_2b = ...  # b) the smallest number of classes with at least 100 duels
answer_CP2_2c = ...  # c) votes of the first point [A, B, C, D, E]
answer_CP2_2d = ...  # d) the class predicted for the first point (a letter)
answer_CP2_2e = ...  # e) votes of the second point [A, B, C, D, E]
answer_CP2_2f = ...  # f) the class predicted with mylearn's rule (a letter)
answer_CP2_2g = ...  # g) the class predicted with the duel rule (a letter)

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP2_2a, answer_CP2_2b, answer_CP2_2c, answer_CP2_2d, answer_CP2_2e, answer_CP2_2f, answer_CP2_2g]):
        RESULTS[f"CP2.2{letter}"] = wb.check(f"CP2.2{letter}", answer)

**CP2.3 — Une itération de k-means et l'inertie obtenue**

In [ ]:
# CP2.3: the values written on your answer sheet
answer_CP2_3a = ...  # a) the cluster (1 or 2) of P1 to P6 after the first assignment
answer_CP2_3b = ...  # b) the inertia with the initial centres
answer_CP2_3c = ...  # c) the new centres [[x1, y1], [x2, y2]]
answer_CP2_3d = ...  # d) the inertia with these centres, same assignment, 2 decimals
answer_CP2_3e = ...  # e) the number (1 to 6) of the point that changes cluster
answer_CP2_3f = ...  # f) the centres after the second update
answer_CP2_3g = ...  # g) the final inertia
answer_CP2_3h = ...  # h) True / False (ou "vrai" / "faux")
answer_CP2_3i = ...  # i) probability that the second centre is P4, P5 or P6 (3 decimals)

if exam_over():
    for letter, answer in zip("abcdefghi", [answer_CP2_3a, answer_CP2_3b, answer_CP2_3c, answer_CP2_3d, answer_CP2_3e, answer_CP2_3f, answer_CP2_3g, answer_CP2_3h, answer_CP2_3i]):
        RESULTS[f"CP2.3{letter}"] = wb.check(f"CP2.3{letter}", answer)

**CP2.4 — Densité d'échantillons et hyper-orange**

In [ ]:
# CP2.4: the values written on your answer sheet
answer_CP2_4a = ...  # a) [density in dimension 3, in dimension 6], 3 decimals
answer_CP2_4b = ...  # b) number of samples for a density of 2 in dimension 8
answer_CP2_4c = ...  # c) the smallest dimension where the density falls below 0.01
answer_CP2_4d = ...  # d) radius of the orange in dimension 9
answer_CP2_4e = ...  # e) the smallest dimension where the orange leaves the box
answer_CP2_4f = ...  # f) share of the volume in the 5 % skin in dimension 30, 3 decimals

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP2_4a, answer_CP2_4b, answer_CP2_4c, answer_CP2_4d, answer_CP2_4e, answer_CP2_4f]):
        RESULTS[f"CP2.4{letter}"] = wb.check(f"CP2.4{letter}", answer)

**CP2.5 — Plan d'évaluation**

In [ ]:
# CP2.5: the values written on your answer sheet
answer_CP2_5a = ...  # a) size of the test set
answer_CP2_5b = ...  # b) sizes of the 5 folds, in the order of the folds
answer_CP2_5c = ...  # c) total number of trainings
answer_CP2_5d = ...  # d) total number of trainings with the nested cross-validation
answer_CP2_5e = ...  # e) [standard error, half-width 1.96 SE], 3 decimals
answer_CP2_5f = ...  # f) the numbers of the protocols with a leak (a list)

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP2_5a, answer_CP2_5b, answer_CP2_5c, answer_CP2_5d, answer_CP2_5e, answer_CP2_5f]):
        RESULTS[f"CP2.5{letter}"] = wb.check(f"CP2.5{letter}", answer)

**CP2.6 — Ridge en dimension 1**

In [ ]:
# CP2.6: the values written on your answer sheet
answer_CP2_6a = ...  # a) [w, b] for lambda = 0
answer_CP2_6b = ...  # b) [w, b] for lambda = 5
answer_CP2_6c = ...  # c) [sum of squared residuals for lambda = 0, for lambda = 5]
answer_CP2_6d = ...  # d) prediction of the new model for x' = 50, 3 decimals

if exam_over():
    for letter, answer in zip("abcd", [answer_CP2_6a, answer_CP2_6b, answer_CP2_6c, answer_CP2_6d]):
        RESULTS[f"CP2.6{letter}"] = wb.check(f"CP2.6{letter}", answer)

**CP2.7 — Trois paires de courbes d'apprentissage (la lecture de c se note avec le corrigé)**

In [ ]:
# CP2.7: the values written on your answer sheet
answer_CP2_7a = ...  # a) the panel (1, 2 or 3) of the underfitting
answer_CP2_7b = ...  # b) the panel of the overfitting with few examples
answer_CP2_7d = ...  # d) the letter of the action for panel 2
answer_CP2_7e = ...  # e) True / False (ou "vrai" / "faux")

if exam_over():
    for letter, answer in zip("abde", [answer_CP2_7a, answer_CP2_7b, answer_CP2_7d, answer_CP2_7e]):
        RESULTS[f"CP2.7{letter}"] = wb.check(f"CP2.7{letter}", answer)

**CP2.8 — Perceptron : NAND, puis XOR**

In [ ]:
# CP2.8: the values written on your answer sheet
answer_CP2_8a = ...  # a) the four values of z during the first epoch
answer_CP2_8b = ...  # b) [w1, w2, b] after the first epoch
answer_CP2_8c = ...  # c) the number of corrections during the first epoch
answer_CP2_8d = ...  # d) the number of inputs well classified with the weights of b)
answer_CP2_8e = ...  # e) [w1, w2, b] after the second epoch

if exam_over():
    for letter, answer in zip("abcde", [answer_CP2_8a, answer_CP2_8b, answer_CP2_8c, answer_CP2_8d, answer_CP2_8e]):
        RESULTS[f"CP2.8{letter}"] = wb.check(f"CP2.8{letter}", answer)

**CP2.9 — Syllogismes et sophismes**

In [ ]:
# CP2.9: the values written on your answer sheet
answer_CP2_9a = ...  # a) six letters S, V or N, in order (for example "SSSSSS")
answer_CP2_9b = ...  # b) the letters of the fallacies of the non-valid arguments, in order
answer_CP2_9c = ...  # c) True / False (ou "vrai" / "faux")

if exam_over():
    for letter, answer in zip("abc", [as_letters(answer_CP2_9a), as_letters(answer_CP2_9b), answer_CP2_9c]):
        RESULTS[f"CP2.9{letter}"] = wb.check(f"CP2.9{letter}", answer)

**CP2.13 — Parties antérieures : matrice de confusion, Bayes et entropie**

In [ ]:
# CP2.13: the values written on your answer sheet
answer_CP2_13a = ...  # a) [TP, FP, FN, TN]
answer_CP2_13b = ...  # b) [precision, recall, F1], 3 decimals
answer_CP2_13c = ...  # c) accuracy, 2 decimals
answer_CP2_13d = ...  # d) P(spam | flagged) with 40 % of spams, 3 decimals
answer_CP2_13e = ...  # e) entropy of the first cluster in bits, 3 decimals
answer_CP2_13f = ...  # f) entropy of the second cluster in bits

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP2_13a, answer_CP2_13b, answer_CP2_13c, answer_CP2_13d, answer_CP2_13e, answer_CP2_13f]):
        RESULTS[f"CP2.13{letter}"] = wb.check(f"CP2.13{letter}", answer)

### Bilan

In [ ]:
# Summary of the automatic checks
import re


def natural(sub_id):
    """CP2.2a before CP2.13a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP2\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    checked = [ok for props in PROPERTIES.values() for _, ok in props]
    print(f"Propriétés du code (CP2.10, CP2.11) : {sum(checked)} sur {len(checked)}.")
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; les explications et les démonstrations (CP2.1, CP2.3 j, CP2.6 1 à 3, d et e, CP2.7 f, CP2.8 f et g, CP2.10 a et d, CP2.12) ; les lectures de la figure de CP2.7 (c) ; et, pour CP2.10 et CP2.11, les nombres recopiés sur ta feuille (seules tes fonctions sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Ensuite : la remédiation du corrigé (relis la synthèse, `05_synthese.md`, pour les notions ratées), puis le mini-projet.